In [149]:
# =============================================================================
# cnn_extractor_kaggle.ipynb
# CNN feature extraction pipeline — Kaggle version.
# Dataset is already attached, no downloading needed.
#
# SETUP (do once before running):
# 1. Go to kaggle.com/datasets/sohangundoju/mirflickr-1m
# 2. Click "New Notebook"
# 3. Settings (right sidebar) → Accelerator → GPU T4 x2
# 4. Only change PACK_NUMBER in Cell 1 each session
# =============================================================================
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — CONFIGURATION (ONLY CHANGE PACK_NUMBER EACH SESSION)
# ─────────────────────────────────────────────────────────────────────────────

PACK_NUMBER     = 10       # ← Change this to 1, 2, 3 ... 10 each session
BATCH_SIZE      = 64      # do not change
PCA_COMPONENTS  = 128     # do not change

KAGGLE_OUTPUT   = '/kaggle/working'
IMAGES_PER_PACK = 100_000

# Each pack has its own folder: images0 = Pack 1, images1 = Pack 2, etc.
KAGGLE_INPUT = f'/kaggle/input/datasets/sohangundoju/mirflickr-1m/images{PACK_NUMBER - 1}/images'

In [150]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────────────────────

!pip install h5py tqdm scikit-learn --quiet

In [151]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

import os
import glob
import pickle
import shutil

import numpy as np
import h5py
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
from tqdm import tqdm
from sklearn.decomposition import PCA

print("✓ All imports successful")

✓ All imports successful


In [152]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — CHECK GPU
# If this prints CPU, go to Settings → Accelerator → GPU T4 x2 → Save
# then re-run from Cell 1.
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cpu':
    raise RuntimeError(
        "No GPU detected. Go to Settings (right sidebar) → "
        "Accelerator → GPU T4 x2 → Save. Then re-run from Cell 1."
    )

print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
print(f"  VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✓ GPU detected: Tesla T4
  VRAM available: 15.6 GB


In [153]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — SET UP OUTPUT PATHS
# All outputs go to /kaggle/working/ which persists after the session.
# Download files from the Output panel (right sidebar) after each session.
# ─────────────────────────────────────────────────────────────────────────────

CNN_OUT_DIR = os.path.join(KAGGLE_OUTPUT, 'features', 'cnn')
PCA_OUT_DIR = os.path.join(KAGGLE_OUTPUT, 'pca')

os.makedirs(CNN_OUT_DIR, exist_ok=True)
os.makedirs(PCA_OUT_DIR, exist_ok=True)

output_h5  = os.path.join(CNN_OUT_DIR, f'pack_{PACK_NUMBER}_cnn.h5')
output_pkl = os.path.join(CNN_OUT_DIR, f'pack_{PACK_NUMBER}_image_paths.pkl')

print(f"✓ Output paths set.")
print(f"  CNN output : {CNN_OUT_DIR}")
print(f"  PCA output : {PCA_OUT_DIR}")

✓ Output paths set.
  CNN output : /kaggle/working/features/cnn
  PCA output : /kaggle/working/pca


In [154]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — FIND IMAGES FOR THIS PACK
# No slicing needed — each pack folder contains exactly that pack's images.
# ─────────────────────────────────────────────────────────────────────────────

print(f"Scanning Pack {PACK_NUMBER} at {KAGGLE_INPUT} ...")

all_patterns = ['**/*.jpg', '**/*.jpeg', '**/*.png']
all_image_paths = []
for pattern in all_patterns:
    all_image_paths.extend(
        glob.glob(os.path.join(KAGGLE_INPUT, pattern), recursive=True)
    )

# Absolute paths, deduplicate, sort — IDENTICAL sort to Aliza and Mahnoor
PACK_PATHS = sorted(list(set([os.path.abspath(p) for p in all_image_paths])))

pack_start = (PACK_NUMBER - 1) * IMAGES_PER_PACK
pack_end   = pack_start + len(PACK_PATHS)

print(f"✓ Pack {PACK_NUMBER}:")
print(f"  Images  : {len(PACK_PATHS)}")
print(f"  ID range: {pack_start} → {pack_end - 1}")
print(f"  First   : {os.path.basename(PACK_PATHS[0])}")
print(f"  Last    : {os.path.basename(PACK_PATHS[-1])}")

if len(PACK_PATHS) == 0:
    raise ValueError(
        f"No images found at {KAGGLE_INPUT}. "
        "Check that the dataset is attached and PACK_NUMBER is correct."
    )

if len(PACK_PATHS) < 90_000:
    print(f"  WARNING: Only {len(PACK_PATHS)} images found — expected ~100,000.")

Scanning Pack 10 at /kaggle/input/datasets/sohangundoju/mirflickr-1m/images9/images ...
✓ Pack 10:
  Images  : 100000
  ID range: 900000 → 999999
  First   : 900000.jpg
  Last    : 999999.jpg


In [155]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — SYNC CHECK
# Share this output in the group chat.
# Aliza and Mahnoor must show identical filenames at the same indices.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print(f"  SYNC CHECK — Pack {PACK_NUMBER} — share this in group chat")
print("=" * 60)
print(f"  Pack images      : {len(PACK_PATHS)}")
print(f"  ID range         : {pack_start} → {pack_end - 1}")
print(f"  First file       : {os.path.basename(PACK_PATHS[0])}")
print(f"  Last  file       : {os.path.basename(PACK_PATHS[-1])}")
if len(PACK_PATHS) > 500:
    print(f"  File #500        : {os.path.basename(PACK_PATHS[500])}")
if len(PACK_PATHS) > 50000:
    print(f"  File #50000      : {os.path.basename(PACK_PATHS[50000])}")
print("=" * 60)
print("  Aliza and Mahnoor must see identical filenames at same indices.")
print("  If any filename differs — stop and debug before extracting.")
print("=" * 60)

  SYNC CHECK — Pack 10 — share this in group chat
  Pack images      : 100000
  ID range         : 900000 → 999999
  First file       : 900000.jpg
  Last  file       : 999999.jpg
  File #500        : 900500.jpg
  File #50000      : 950000.jpg
  Aliza and Mahnoor must see identical filenames at same indices.
  If any filename differs — stop and debug before extracting.


In [156]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — SHARED IMAGE LOADING UTILITY
# DO NOT MODIFY ANYTHING IN THIS CELL.
# Aliza and Mahnoor have the exact same cell in their notebooks.
# ─────────────────────────────────────────────────────────────────────────────

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_single_image(path: str) -> np.ndarray:
    """
    Loads and preprocesses one image for ResNet-50.
    Returns float32 array of shape (3, 224, 224).
    Raises exception on failure — handled by caller.
    """
    img = Image.open(path).convert('RGB')
    img = img.resize((224, 224), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    arr /= 255.0
    arr -= IMAGENET_MEAN
    arr /= IMAGENET_STD
    arr  = arr.transpose(2, 0, 1)   # (H,W,C) → (C,H,W)
    return arr


def load_paths_in_batches(image_paths: list, id_offset: int, batch_size: int):
    """
    Generator — yields (global_ids, image_arrays) until all paths exhausted.
    Failed images → zero array so all files stay the same length.
    """
    failed_log = f"/kaggle/working/failed_pack_{PACK_NUMBER}.txt"
    total      = len(image_paths)

    print(f"[loader] {total} images, IDs {id_offset} → {id_offset + total - 1}")

    for batch_start in range(0, total, batch_size):
        batch_paths       = image_paths[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_paths)
        global_ids        = list(range(id_offset + batch_start,
                                       id_offset + batch_start + actual_batch_size))
        image_arrays      = np.zeros((actual_batch_size, 3, 224, 224), dtype=np.float32)

        for i, path in enumerate(batch_paths):
            try:
                image_arrays[i] = load_single_image(path)
            except Exception as e:
                with open(failed_log, 'a') as f:
                    f.write(f"{path}\t{e}\n")

        yield global_ids, image_arrays

print("✓ Shared loading utility defined.")


✓ Shared loading utility defined.


In [157]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — LOAD RESNET-50
# ─────────────────────────────────────────────────────────────────────────────

model = models.resnet50(pretrained=True)
model.fc = nn.Identity()   # remove classification head → outputs 2048-dim embeddings
model.eval()
model.to(DEVICE)

print(f"✓ ResNet-50 loaded on {DEVICE}.")
print(f"  Output dimension: 2048 (before PCA)")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✓ ResNet-50 loaded on cuda.
  Output dimension: 2048 (before PCA)


In [158]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — RUN BATCH INFERENCE
# ─────────────────────────────────────────────────────────────────────────────

all_embeddings = []
all_ids        = []
id_offset      = pack_start   # global ID of first image in this pack

with tqdm(total=len(PACK_PATHS), desc=f"CNN Pack {PACK_NUMBER}", unit="img") as pbar:
    for global_ids, image_arrays in load_paths_in_batches(PACK_PATHS, id_offset, BATCH_SIZE):

        batch_tensor = torch.from_numpy(image_arrays).to(DEVICE)

        with torch.no_grad():
            embeddings = model(batch_tensor)   # (batch_size, 2048)

        embeddings_np = embeddings.cpu().numpy()
        all_embeddings.append(embeddings_np)
        all_ids.extend(global_ids)
        pbar.update(len(global_ids))

all_embeddings = np.concatenate(all_embeddings, axis=0)   # (N, 2048)
all_ids        = np.array(all_ids, dtype=np.int64)        # (N,)

print(f"\n✓ Inference complete.")
print(f"  Embeddings shape : {all_embeddings.shape}")
print(f"  ID range         : {all_ids[0]} → {all_ids[-1]}")

CNN Pack 10:   0%|          | 0/100000 [00:00<?, ?img/s]

[loader] 100000 images, IDs 900000 → 999999


CNN Pack 10: 100%|██████████| 100000/100000 [28:29<00:00, 58.48img/s]



✓ Inference complete.
  Embeddings shape : (100000, 2048)
  ID range         : 900000 → 999999


In [159]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — PCA COMPRESSION (2048 → 128 dimensions)
#
# PACK 1: fit PCA and save pca_model.pkl to /kaggle/working/pca/
#         → download this file from the Output panel after Pack 1 is done
#         → for Pack 2+: upload pca_model.pkl as a Kaggle dataset (see below)
#
# PACK 2-10: attach your pca_model dataset to this notebook
#            Settings → Add data → Your datasets → select pca_model
#            Then the model loads from /kaggle/input/pca-model/pca_model.pkl
# ─────────────────────────────────────────────────────────────────────────────

pca_model_path_out  = os.path.join(PCA_OUT_DIR, 'pca_model.pkl')
pca_model_path_in   = '/kaggle/input/datasets/piranchalghai/pca-model/pca_model.pkl'  # for Pack 2-10

if PACK_NUMBER == 1:
    print("Pack 1 — fitting PCA ...")
    pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
    embeddings_compressed = pca.fit_transform(all_embeddings)

    with open(pca_model_path_out, 'wb') as f:
        pickle.dump(pca, f)

    explained = np.sum(pca.explained_variance_ratio_) * 100
    print(f"✓ PCA fitted and saved to {pca_model_path_out}")
    print(f"  Explained variance: {explained:.1f}%")
    print(f"\n  ► IMPORTANT: After this session ends, download pca_model.pkl")
    print(f"    from the Output panel (right sidebar → Output tab).")
    print(f"    Then upload it to Kaggle as a dataset named 'pca-model'")
    print(f"    so Packs 2-10 can load it.")

else:
    if not os.path.exists(pca_model_path_in):
        raise FileNotFoundError(
            f"PCA model not found at {pca_model_path_in}.\n"
            "You need to:\n"
            "  1. Upload pca_model.pkl (saved during Pack 1) as a Kaggle dataset\n"
            "  2. Name the dataset 'pca-model'\n"
            "  3. Attach it to this notebook via Settings → Add data\n"
            "  4. Re-run from Cell 1"
        )

    print(f"Pack {PACK_NUMBER} — loading PCA model from {pca_model_path_in} ...")
    with open(pca_model_path_in, 'rb') as f:
        pca = pickle.load(f)

    embeddings_compressed = pca.transform(all_embeddings)
    print(f"✓ PCA transform applied.")

embeddings_compressed = embeddings_compressed.astype(np.float32)
print(f"  Compressed shape : {embeddings_compressed.shape}")

Pack 10 — loading PCA model from /kaggle/input/datasets/piranchalghai/pca-model/pca_model.pkl ...
✓ PCA transform applied.
  Compressed shape : (100000, 128)


In [160]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — SAVE EMBEDDINGS TO HDF5
# ─────────────────────────────────────────────────────────────────────────────

with h5py.File(output_h5, 'w') as f:
    f.create_dataset('embeddings', data=embeddings_compressed,
                     dtype=np.float32, compression='gzip', compression_opts=4)
    f.create_dataset('image_ids',  data=all_ids, dtype=np.int64)
    f.attrs['pack_number']   = PACK_NUMBER
    f.attrs['n_images']      = len(all_ids)
    f.attrs['embedding_dim'] = PCA_COMPONENTS
    f.attrs['id_offset']     = int(all_ids[0])

print(f"✓ HDF5 saved: {output_h5}")
print(f"  File size: {os.path.getsize(output_h5) / 1e6:.1f} MB")

✓ HDF5 saved: /kaggle/working/features/cnn/pack_10_cnn.h5
  File size: 48.2 MB


In [162]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — SAVE IMAGE PATH MAPPING
# ─────────────────────────────────────────────────────────────────────────────

id_to_path = {pack_start + i: path for i, path in enumerate(PACK_PATHS)}

with open(output_pkl, 'wb') as f:
    pickle.dump(id_to_path, f)

print(f"✓ Image path mapping saved: {output_pkl}")
print(f"  Entries : {len(id_to_path)}")
print(f"  Sample  : ID {pack_start} → {id_to_path[pack_start]}")

✓ Image path mapping saved: /kaggle/working/features/cnn/pack_10_image_paths.pkl
  Entries : 100000
  Sample  : ID 900000 → /kaggle/input/datasets/sohangundoju/mirflickr-1m/images9/images/90/900000.jpg


In [163]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — VERIFY OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

with h5py.File(output_h5, 'r') as f:
    emb_shape = f['embeddings'].shape
    ids_shape = f['image_ids'].shape
    emb_min   = float(f['embeddings'][0].min())
    emb_max   = float(f['embeddings'][0].max())
    first_id  = int(f['image_ids'][0])
    last_id   = int(f['image_ids'][-1])

assert emb_shape[0] == ids_shape[0],   "Mismatch: embeddings and IDs different lengths"
assert emb_shape[1] == PCA_COMPONENTS, f"Wrong embedding dim: {emb_shape[1]}"
assert first_id == pack_start,         f"Wrong first ID: {first_id}, expected {pack_start}"
assert not (emb_min == 0 and emb_max == 0), "All-zero embeddings — something went wrong"

print("✓ Verification passed.")
print(f"  Embeddings shape : {emb_shape}")
print(f"  ID range         : {first_id} → {last_id}")
print(f"  Value range      : [{emb_min:.4f}, {emb_max:.4f}]")
print()
print("=" * 60)
print(f"  Pack {PACK_NUMBER} COMPLETE")
print(f"  Files saved to /kaggle/working/features/cnn/")
print()
print(f"  ► Download these files from the Output panel now:")
print(f"    pack_{PACK_NUMBER}_cnn.h5")
print(f"    pack_{PACK_NUMBER}_image_paths.pkl")
if PACK_NUMBER == 1:
    print(f"    pca/pca_model.pkl  ← CRITICAL: needed for Packs 2-10")
print()
print(f"  ► Change PACK_NUMBER to {PACK_NUMBER + 1} for next session")
print("=" * 60)

✓ Verification passed.
  Embeddings shape : (100000, 128)
  ID range         : 900000 → 999999
  Value range      : [-5.1757, 5.6840]

  Pack 10 COMPLETE
  Files saved to /kaggle/working/features/cnn/

  ► Download these files from the Output panel now:
    pack_10_cnn.h5
    pack_10_image_paths.pkl

  ► Change PACK_NUMBER to 11 for next session


In [164]:
#─────────────────────────────────────────────────────────────────────────────
# CELL 15 — MERGE (RUN ONCE AFTER ALL 10 PACKS ARE DONE)
#
# Before running this cell:
# - All 10 pack_N_cnn.h5 files must be uploaded as a Kaggle dataset
#   named 'mirflickr-cnn-packs' with all files inside
# - Attach it via Settings → Add data → Your datasets
# - Files will be at /kaggle/input/mirflickr-cnn-packs/
# ─────────────────────────────────────────────────────────────────────────────

PACKS_INPUT_DIR = '/kaggle/input/datasets/piranchalghai/mirflickrcnnpacks'

print("Checking all 10 pack files exist before merging ...")
missing = []
for n in range(1, 11):
    path = os.path.join(PACKS_INPUT_DIR, f'pack_{n}_cnn.h5')
    if not os.path.exists(path):
        missing.append(n)

if missing:
    raise FileNotFoundError(
        f"Missing pack files for packs: {missing}.\n"
        "Upload all 10 pack_N_cnn.h5 files as a Kaggle dataset first."
    )

print("✓ All 10 pack files found. Starting merge ...")

all_emb_list = []
all_ids_list = []
all_paths    = {}

for n in range(1, 11):
    h5_path  = os.path.join(PACKS_INPUT_DIR, f'pack_{n}_cnn.h5')
    pkl_path = os.path.join(PACKS_INPUT_DIR, f'pack_{n}_image_paths.pkl')

    with h5py.File(h5_path, 'r') as f:
        all_emb_list.append(f['embeddings'][:])
        all_ids_list.append(f['image_ids'][:])

    with open(pkl_path, 'rb') as f:
        all_paths.update(pickle.load(f))

    print(f"  Pack {n} loaded.")

merged_embeddings = np.concatenate(all_emb_list, axis=0)
merged_ids        = np.concatenate(all_ids_list, axis=0)

print(f"\n✓ Merge complete.")
print(f"  Total embeddings : {merged_embeddings.shape}")
print(f"  ID range         : {merged_ids[0]} → {merged_ids[-1]}")

merged_h5_path  = '/kaggle/working/all_cnn_embeddings.h5'
merged_pkl_path = '/kaggle/working/all_image_paths.pkl'

with h5py.File(merged_h5_path, 'w') as f:
    f.create_dataset('embeddings', data=merged_embeddings,
                     dtype=np.float32, compression='gzip', compression_opts=4)
    f.create_dataset('image_ids',  data=merged_ids, dtype=np.int64)
    f.attrs['total_images']  = len(merged_ids)
    f.attrs['embedding_dim'] = PCA_COMPONENTS

with open(merged_pkl_path, 'wb') as f:
    pickle.dump(all_paths, f)

print(f"\n✓ Saved merged files to /kaggle/working/:")
print(f"  all_cnn_embeddings.h5  ({os.path.getsize(merged_h5_path)  / 1e6:.0f} MB)")
print(f"  all_image_paths.pkl    ({os.path.getsize(merged_pkl_path) / 1e6:.0f} MB)")
print(f"\n  ► Download both files and share with the team.")

Checking all 10 pack files exist before merging ...
✓ All 10 pack files found. Starting merge ...
  Pack 1 loaded.
  Pack 2 loaded.
  Pack 3 loaded.
  Pack 4 loaded.
  Pack 5 loaded.
  Pack 6 loaded.
  Pack 7 loaded.
  Pack 8 loaded.
  Pack 9 loaded.
  Pack 10 loaded.

✓ Merge complete.
  Total embeddings : (1000000, 128)
  ID range         : 0 → 999999

✓ Saved merged files to /kaggle/working/:
  all_cnn_embeddings.h5  (482 MB)
  all_image_paths.pkl    (85 MB)

  ► Download both files and share with the team.
